# Preprocessing target data for use with Abil.py
Target (the variable you want to model) data needs to be put on the same coordinate grid as your predictor (environmental) data.
You'll need to load your observations, do any additional data cleaning (dropping certain methods, any out of range data), and then convert the coordinates to match the predictor data.

### Import required packages

In [1]:
import pandas as pd
import numpy as np
import xarray as xr

### Define your observation data filename and variable of interest

In [2]:
filename = '/home/mv23682/Documents/Abil_Wiseman2025/scripts/data/Marsh_et_al_2025_Database_V_1.0.csv'
varname = 'Primary_Production'

### Load data
I'm using usecols here to specifically extract the columns I need, as this is a large spreadsheet and contains a lot of extra information that I'd just drop later

In [3]:
d_raw = pd.read_csv(filename,usecols=['Date of collection','Latitude','Longitude','Sampling Depth [m]','Primary Production (PP) [µmol C m-3 d-1]'])

In [4]:
# Extract month value from date of sample
d_raw['DateTime'] = pd.to_datetime(d_raw['Date of collection'], dayfirst=False, errors='coerce')
d_raw['Month'] = d_raw['DateTime'].dt.month

# Rename columns to match predictor data
d = d_raw.rename(columns={
    "Latitude":"lat", 
    "Longitude":"lon", 
    "Sampling Depth [m]":"depth", 
    "Primary Production (PP) [µmol C m-3 d-1]":"Primary_Production", 
    "Month":"time"})


# Subset dataframe to only necessary variable
d = d[['lat','lon','depth','time','Primary_Production']]

# Drop any rows with missing values
d.dropna(subset=["Primary_Production"], inplace=True)


In [5]:
# Bin data to match grid of predictor data (180x360x41x12)
# Grid to 180×360×41×12
depth_bins = np.linspace(0, 205, 42)
depth_labels = np.linspace(0, 200, 41)
d["depth"] = pd.cut(d["depth"], depth_bins, labels=depth_labels, include_lowest=True).astype(float)

lat_bins = np.linspace(-90, 90, 181)
lat_labels = np.linspace(-90, 89, 180)
d["lat"] = pd.cut(d["lat"], lat_bins, labels=lat_labels).astype(float)

lon_bins = np.linspace(-180, 180, 361)
lon_labels = np.linspace(-180, 179, 360)
d["lon"] = pd.cut(d["lon"], lon_bins, labels=lon_labels).astype(float)


# Group data so there is only one observation per gridcell
d = d.groupby(["lat", "lon", "depth", "time"]).mean().reset_index()
d.set_index(["lat", "lon", "depth", "time"], inplace=True)


### Load environmental predictor data

In [6]:
# Load environment data
ds_env = xr.open_dataset('/home/mv23682/Documents/Abil_tutorial/data/env_data_na.nc')
df_env = ds_env.to_dataframe().reset_index()
env_cols = ["temperature","no3","o2","PAR"]
df_env = df_env[env_cols + ["lat","lon","depth","time"]].set_index(["lat","lon","depth","time"])
print(f"Environment rows: {len(df_env)}")

# Filter only rows whose environmental data exists AND is fully non-NaN
valid_env_idx = df_env.dropna().index
d_na = d.loc[d.index.intersection(valid_env_idx)]
print(f"Rows after dropping samples without complete environmental data: {len(d_na)}")

# Merge with environment data
d_na = d_na.join(df_env, how="inner").reset_index()
print(f"Rows after merging with environment: {len(d_na)}")

# Save combined CSV
outfile = "/home/mv23682/Documents/Abil_tutorial/data/training_na.csv"
d_na.to_csv(outfile, index=False)

print("\n=== Finished ===")
print("Saved combined CSV to:")
print(outfile)

Environment rows: 114192
Rows after dropping samples without complete environmental data: 179
Rows after merging with environment: 179

=== Finished ===
Saved combined CSV to:
/home/mv23682/Documents/Abil_tutorial/data/training_na.csv
